# 06 — Analyse des erreurs et recommandations

**Projet** : Détection de fraude sur transactions de paiement avec scikit-learn
**Détecteur évalué** : `isolation_forest` (Forêt d'isolation (Isolation Forest))
**Métrique primaire** : `pr_auc` (seuil cible : 0.42)

Une fraude ne se résume pas à une matrice de confusion : le coût d'une fausse alerte est du temps
analyste, le coût d'une fraude manquée est une perte nette. Ce notebook évalue le détecteur **au
point de fonctionnement réel** (le budget d'investigation), puis dissèque ses erreurs pour en tirer
des recommandations actionnables.

Les étiquettes utilisées ici (`is_fraud`, `fraud_scheme`) sont des **métadonnées** : le modèle ne les a
jamais vues, ni à l'entraînement ni dans la matrice de features.

## Objectifs pédagogiques

1. Évaluer le détecteur sur le split de test **hors échantillon** et comparer au plancher aléatoire.
1. Traduire le score en **décision de capacité** : table d'arbitrage volume d'alertes → rappel / précision / lift.
1. Vérifier la **couverture par mode opératoire** : un bon score global peut masquer un schéma non détecté.
1. Disséquer les **erreurs** (fraudes manquées, fausses alertes) et les **facteurs contributifs**, puis formuler des recommandations chiffrées.

**Objectifs transverses du dépôt**

- Comprendre pourquoi la fraude se détecte sans supervision : étiquette tardive (chargeback à J+30), partielle et biaisée par les règles existantes.
- Construire un pipeline sans fuite : `is_fraud` et `fraud_scheme` sont des métadonnées exclues des features par configuration.
- Lire les bonnes métriques en forte imbalance : PR AUC et lift plutôt qu'accuracy et ROC AUC, et toujours relativement au plancher (= prévalence).

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (12000 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 12000

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.42)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

In [ ]:
from src.models import build_model

# Tâche non supervisée : `fit` reçoit `None` en guise de cible. Le modèle apprend la région de
# densité « normale » sur le train uniquement ; la validation sert à choisir le point de
# fonctionnement, jamais à ajuster les hyperparamètres sur les fraudes.
MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
FIT_RESULT = MODEL.fit(PREPARED["X_train"], None, X_val=PREPARED["X_val"], y_val=None, callbacks=[])
print(MODEL.summary())
print(f"entraînement : {FIT_RESULT.duration_seconds:.2f} s")

## 1. Évaluation hors échantillon

In [ ]:
from src.evaluation.evaluator import Evaluator

EVALUATOR = Evaluator.from_config(MODEL, CONFIG.model_dump(), NB_PATHS)
RESULT = EVALUATOR.evaluate(
    PREPARED["X_test"],
    None,
    split="test",
    context=PREPARED["enriched"]["test"],
)

metrics_frame = pd.DataFrame(
    {"métrique": list(RESULT.metrics), "valeur": [RESULT.metrics[name] for name in RESULT.metrics]}
)
print(f"transactions évaluées   : {RESULT.n_samples}")
print(f"fraudes confirmées      : {int(RESULT.extras['n_frauds'])}")
print(f"prévalence              : {RESULT.prevalence:.3%}")
print(f"budget d'investigation  : {RESULT.budget} alertes ({RESULT.extras['budget_rate']:.1%})")
print(f"seuil de score retenu   : {RESULT.threshold:.4f}")
print(f"lift au budget          : {RESULT.lift_at_budget:.1f}x")
print(f"plancher (aléatoire)    : {float(RESULT.extras['baseline']['random_pr_auc']):.4f}")
display(metrics_frame.round(4))

**Ce qu'il faut retenir**

- La PR AUC se lit **relativement à la prévalence** : à 1,8 % de fraude, un plancher de 0,018 signifie qu'une PR AUC de 0,40 représente un gain d'un facteur ~22, et non « 40 % de réussite ».
- La ROC AUC reste élevée même quand la file d'alertes est noyée sous les faux positifs : c'est pourquoi elle est secondaire ici, et la PR AUC primaire.
- Le lift au budget est la traduction métier du classement : à capacité égale, combien de fois plus de fraude qu'un tirage aléatoire.

## 2. Le point de fonctionnement : arbitrer au budget d'investigation

In [ ]:
budget_table = pd.DataFrame(RESULT.curves["budget_tradeoff"])
display(budget_table.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.2))
axes[0].plot(
    budget_table["budget_rate"] * 100,
    budget_table["recall"],
    marker="o",
    ms=4,
    color="#0a9396",
    label="rappel",
)
axes[0].plot(
    budget_table["budget_rate"] * 100,
    budget_table["precision"],
    marker="s",
    ms=4,
    color="#ae2012",
    label="précision",
)
axes[0].axvline(
    float(RESULT.extras["budget_rate"]) * 100,
    color="#ee9b00",
    ls="--",
    lw=1.6,
    label=f"budget retenu ({RESULT.extras['budget_rate']:.0%})",
)
axes[0].set_xlabel("volume d'alertes (% du flux)")
axes[0].set_ylabel("rappel / précision")
axes[0].set_ylim(0, 1.02)
axes[0].set_title("Arbitrage capacité d'analyse ↔ fraude capturée")
axes[0].legend(fontsize=8)

axes[1].plot(
    budget_table["alerts"],
    budget_table["frauds_captured"],
    marker="o",
    ms=4,
    color="#0a9396",
    label="détecteur",
)
random_capture = budget_table["alerts"] * float(RESULT.prevalence)
axes[1].plot(
    budget_table["alerts"], random_capture, ls="--", color="#ae2012", label="tirage aléatoire"
)
axes[1].set_xlabel("nombre d'alertes traitées")
axes[1].set_ylabel("fraudes capturées")
axes[1].set_title("Courbe de capture : le classement fait le gain")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Le seuil ne se choisit pas sur un F1 abstrait mais sur la **capacité réelle** de l'équipe : la colonne « fausses alertes » est un coût en jours-analystes.
- La courbe de capture montre le gain du classement : à 2 % du flux inspecté, un bon détecteur capture une part de la fraude très supérieure à la diagonale aléatoire.
- Doubler le volume d'alertes achète du rappel au prix d'une précision qui chute : c'est un arbitrage de direction, pas un réglage de modèle.

## 3. Couverture par mode opératoire

In [ ]:
SCHEME_LABELS = {
    "card_not_present": "Card not present",
    "account_takeover": "Prise de compte",
    "synthetic_identity": "Identité synthétique",
    "friendly_fraud": "Fraude amicale",
    "legitimate": "Légitime",
}

per_scheme = RESULT.per_scheme
if per_scheme.empty:
    print("Aucune fraude confirmée dans ce split : couverture non mesurable.")
else:
    table = per_scheme.copy()
    table["mode opératoire"] = [
        SCHEME_LABELS.get(str(name), str(name)) for name in table["fraud_scheme"]
    ]
    display(
        table[
            [
                "mode opératoire",
                "frauds",
                "share_of_fraud",
                "captured_at_budget",
                "recall_at_budget",
                "median_rank",
                "best_rank",
                "worst_rank",
            ]
        ].round(3)
    )

    fig, axis = plt.subplots(figsize=(7.6, 3.8))
    ordered = table.sort_values("recall_at_budget")
    axis.barh(
        [SCHEME_LABELS.get(str(name), str(name)) for name in ordered["fraud_scheme"]],
        ordered["recall_at_budget"],
        color="#0a9396",
    )
    for position, (recall, count) in enumerate(
        zip(ordered["recall_at_budget"], ordered["frauds"], strict=True)
    ):
        axis.text(
            float(recall) + 0.01,
            position,
            f"{float(recall):.2f} (n={int(count)})",
            va="center",
            fontsize=8,
        )
    axis.set_xlim(0, 1.15)
    axis.set_xlabel("rappel au budget")
    axis.set_title("Couverture par mode opératoire")
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- Un rappel global de 0,6 peut cacher un rappel de 0,1 sur un schéma minoritaire — et c'est souvent celui qui croît (card testing automatisé, identités synthétiques).
- La fraude amicale est structurellement la plus difficile : la transaction est légitime dans sa forme, seul l'historique de contestations trahit le comportement. Un détecteur transactionnel ne peut pas la voir.
- La surveillance de production doit être **ventilée par schéma**, pas agrégée : sinon une dégradation ciblée passe inaperçue tant que la métrique globale tient.

## 4. Concentration de la fraude : lift par décile

In [ ]:
lift_rows = RESULT.curves.get("lift_by_decile") or []
if not lift_rows:
    print("Lift non calculable sur ce split.")
else:
    lift_table = pd.DataFrame(lift_rows)
    display(lift_table.round(4))

    frauds = pd.Series(RESULT.predictions.loc[RESULT.predictions["is_fraud"] == 1, "rank"])
    captured = np.array(
        [(frauds <= position).sum() for position in range(1, len(RESULT.predictions) + 1)]
    )
    share = captured / max(int(frauds.size), 1)
    inspected = np.arange(1, len(RESULT.predictions) + 1) / len(RESULT.predictions)

    fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.2))
    axes[0].bar(
        [f"D{int(value)}" for value in lift_table["decile"]],
        lift_table["lift"],
        color=["#ee9b00" if int(value) >= 8 else "#0a9396" for value in lift_table["decile"]],
    )
    axes[0].axhline(1.0, color="#ae2012", ls="--", lw=1.2, label="lift = 1 (aléatoire)")
    axes[0].set_ylabel("lift sur la prévalence")
    axes[0].set_title("Concentration de la fraude par décile de score")
    axes[0].legend(fontsize=8)

    axes[1].plot(inspected * 100, share * 100, color="#0a9396", lw=2.0, label="détecteur")
    axes[1].plot([0, 100], [0, 100], color="#ae2012", ls="--", lw=1.2, label="aléatoire")
    axes[1].axvline(
        float(RESULT.extras["budget_rate"]) * 100,
        color="#ee9b00",
        ls="--",
        lw=1.6,
        label="budget retenu",
    )
    axes[1].set_xlabel("part du flux inspectée (%)")
    axes[1].set_ylabel("part de la fraude capturée (%)")
    axes[1].set_title("Courbe de capture cumulative")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

**Ce qu'il faut retenir**

- Le lift du décile supérieur mesure la rentabilité de l'investigation : c'est le chiffre à présenter à la direction des risques.
- Si le lift est proche de 1 dans tous les déciles, le score n'ordonne rien — même si la ROC AUC semble correcte.
- La courbe de capture cumulative sert à dimensionner l'équipe : elle répond directement à « combien d'alertes pour capturer 70 % de la fraude ? ».

## 5. Analyse des erreurs

In [ ]:
rows = RESULT.predictions
errors = RESULT.errors
context_columns = [
    column
    for column in (
        "transaction_id",
        "amount_eur",
        "merchant_category",
        "channel",
        "transactions_24h",
        "failed_attempts_1h",
        "device_age_days",
        "three_ds_authenticated",
        "amount_to_customer_avg_ratio",
        "previous_chargebacks_12m",
    )
    if column in errors.columns
]

outcomes = rows["outcome"].value_counts(normalize=True) * 100
print("Répartition des décisions au budget :")
display(outcomes.round(2).to_frame("%"))

for kind, title in (
    ("false_negative", "Fraudes manquées les mieux classées (marge de progrès)"),
    ("false_positive", "Fausses alertes les plus convaincantes (coût analyste)"),
    ("true_positive", "Fraudes capturées (ce que le détecteur sait faire)"),
):
    subset = errors[errors["error_kind"] == kind].head(8)
    if subset.empty:
        continue
    print(f"\n### {title}")
    display(subset[["rank", "score", "fraud_scheme", *context_columns]].round(3))

# Les fausses alertes sont-elles des outliers légitimes ? On vérifie sur les colonnes brutes.
false_alarms = rows[rows["outcome"] == "false_positive"]
if not false_alarms.empty and "amount_eur" in PREPARED["enriched"]["test"].columns:
    test_frame = PREPARED["enriched"]["test"].reset_index(drop=True)
    amount = pd.to_numeric(test_frame["amount_eur"], errors="coerce")
    high_value = float((amount.loc[false_alarms.index] > amount.quantile(0.99)).mean())
    print(f"\nPart des fausses alertes dont le montant dépasse le P99 du flux : {high_value:.1%}")

**Ce qu'il faut retenir**

- Les **fraudes manquées les mieux classées** sont la marge de progrès accessible : elles portent une signature presque suffisante. Si elles se concentrent sur un schéma, c'est ce schéma qu'il faut outiller.
- Les **fausses alertes les plus convaincantes** sont souvent des outliers légitimes (achat de luxe, voyageur d'affaires nocturne, paiement d'entreprise). Elles relèvent d'une liste blanche métier, pas d'un re-réglage du modèle.
- Comparer les fraudes capturées aux fraudes manquées, colonne par colonne, identifie la variable qui fait la différence : c'est le prochain feature à construire.

## 6. Facteurs contributifs (importance par permutation)

In [ ]:
importance = RESULT.feature_importance
if importance.empty:
    print("Importance par permutation non calculable sur ce split.")
else:
    top = importance.head(15)
    display(top.round(5))

    fig, axis = plt.subplots(figsize=(7.8, 0.34 * len(top) + 1.8))
    axis.barh(
        top["feature"][::-1],
        top["importance_mean"][::-1],
        xerr=top["importance_std"][::-1],
        color=["#ae2012" if value < 0 else "#0a9396" for value in top["importance_mean"][::-1]],
    )
    axis.axvline(0.0, color="#495057", lw=1.0)
    axis.set_xlabel("importance par permutation (recouvrement du top-budget)")
    axis.set_title("Facteurs contributifs du score d'anomalie")
    fig.tight_layout()
    plt.show()

    print(
        f"\npart de l'importance portée par la première feature : {float(top.iloc[0]['share']):.1%}"
    )
    negative = importance[importance["importance_mean"] < 0]
    print(f"features à importance négative : {list(negative['feature']) or 'aucune'}")

**Ce qu'il faut retenir**

- La permutation mesure la chute du **recouvrement du top-budget** quand une colonne est mélangée : cette définition est identique pour tous les détecteurs, ce qui permet de les comparer.
- Une importance **négative** signifie que permuter la colonne améliore le classement : la feature apporte du bruit et doit être retirée ou retravaillée.
- Une importance concentrée sur une seule variable (> 45 %) rend le détecteur fragile : il se réduit presque à une règle univariée, que les fraudeurs contournent en premier.

## 7. Diagnostics automatiques

In [ ]:
from src.evaluation.reports import ReportBuilder

BUILDER = ReportBuilder(NB_PATHS, config=CONFIG.model_dump())
diagnostics = BUILDER.diagnostics(RESULT)

if not diagnostics:
    print("Aucun signal de faiblesse marqué sur ce split.")
for title, detail in diagnostics:
    print(f"\n### {title}\n{detail}")

print("\n--- seuils du cas d'usage ---")
for name, value in sorted(BUILDER.thresholds.items()):
    print(f"  {name}: {value}")

**Ce qu'il faut retenir**

- Les diagnostics sont produits par le même code que le rapport : une faiblesse détectée ici le sera aussi en production, sans intervention manuelle.
- Les seuils affichés viennent du cas d'usage (`extras.quality` du manifeste) et sont écrasables par `metrics.thresholds` dans la configuration Hydra : aucun chiffre n'est codé en dur.

## 8. Recommandations

In [ ]:
recommendations = BUILDER.recommendations(RESULT)
for index, item in enumerate(recommendations, start=1):
    print(f"{index}. {item}")

**Ce qu'il faut retenir**

- Les recommandations **mesurées** (issues des chiffres du split) précèdent les bonnes pratiques documentées du cas d'usage : c'est ce qui les rend actionnables.
- Trois leviers reviennent systématiquement en détection de fraude : les features de vélocité à fenêtre courte, les indicateurs de manquants, et la liste blanche des outliers légitimes.
- Le détecteur ne remplace pas les règles métier : il couvre les schémas inédits, elles couvrent les obligations réglementaires (LCB-FT). Les deux se combinent dans un classement unique.

## 9. Génération du rapport d'évaluation

In [ ]:
artifacts = BUILDER.build(RESULT, model=MODEL)
for kind, path in sorted(artifacts.items()):
    relative = path.relative_to(PROJECT_ROOT) if path.is_absolute() else path
    print(f"{kind:<18} -> {relative}")

display(Markdown(f"### Rapport généré\n\n`{artifacts['report']}`"))
if "figure_pr_curve" in artifacts:
    display(Image(filename=str(artifacts["figure_pr_curve"]), width=720))
if "figure_budget_tradeoff" in artifacts:
    display(Image(filename=str(artifacts["figure_budget_tradeoff"]), width=760))

**Ce qu'il faut retenir**

- Le rapport Markdown, son équivalent JSON, les tables CSV (budget, couverture par schéma, erreurs) et les figures sont écrits dans `outputs/notebooks/` pour ne jamais écraser les artefacts de `make train`.
- Le JSON est la source consommable par un tableau de bord ou un registre de modèles (voir `mlops/model-registry`) ; le Markdown est la source lisible par le responsable fraude.
- Rejouer `make evaluate` régénère exactement ces artefacts à partir des modèles entraînés : l'évaluation est reproductible, pas seulement l'entraînement.